In [7]:
import duckdb
import pandas as pd

# Connect to gold layer
con = duckdb.connect('../data/gold/gold.duckdb')
print("Connected ✓")

Connected ✓


In [8]:
# ── Row count validation ──────────────────────────────────────────────
silver_count = con.execute("SELECT COUNT(*) FROM silver").fetchone()[0]
fact_count = con.execute("SELECT COUNT(*) FROM fact_loans").fetchone()[0]

print(f"Silver rows:    {silver_count:,}")
print(f"Fact rows:      {fact_count:,}")
print(f"Difference:     {silver_count - fact_count:,}")
print(f"Match: {silver_count == fact_count}")

Silver rows:    1,246,701
Fact rows:      1,246,701
Difference:     0
Match: True


In [9]:
# ── Duplicate validation ──────────────────────────────────────────────
dupes = con.execute("""
    SELECT loan_key, COUNT(*) as count
    FROM fact_loans
    GROUP BY loan_key
    HAVING COUNT(*) > 1
""").fetchdf()

print(f"Duplicate loan_keys: {len(dupes)}")

# ── Null validation ──────────────────────────────────────────────
nulls = con.execute("""
    SELECT 
        COUNT(*) - COUNT(loan_key) AS loan_key_nulls,
        COUNT(*) - COUNT(risk_key) AS risk_key_nulls,
        COUNT(*) - COUNT(purpose_key) AS purpose_key_nulls,
        COUNT(*) - COUNT(date_key) AS date_key_nulls,
        COUNT(*) - COUNT(default_flag) AS default_flag_nulls
    FROM fact_loans
""").fetchdf()

print("\nNull counts in key columns:")
print(nulls)

Duplicate loan_keys: 0

Null counts in key columns:
   loan_key_nulls  risk_key_nulls  purpose_key_nulls  date_key_nulls  \
0               0               0                  0               0   

   default_flag_nulls  
0                   0  


In [10]:
# ── Dimension validation ──────────────────────────────────────────────
print("dim_date:")
print(con.execute("SELECT COUNT(*) as rows, MIN(year) as min_year, MAX(year) as max_year FROM dim_date").fetchdf())

print("\ndim_risk:")
print(con.execute("SELECT COUNT(*) as rows, COUNT(DISTINCT sub_grade) as grades, COUNT(DISTINCT term) as terms FROM dim_risk").fetchdf())

print("\ndim_purpose:")
print(con.execute("SELECT COUNT(*) as rows FROM dim_purpose").fetchdf())

print("\ndefault_flag distribution:")
print(con.execute("""
    SELECT default_flag, COUNT(*) as count, ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 2) as pct
    FROM fact_loans
    GROUP BY default_flag
""").fetchdf())

dim_date:
   rows  min_year  max_year
0    77      2012      2018

dim_risk:
   rows  grades  terms
0   823      35      2

dim_purpose:
   rows
0    14

default_flag distribution:
   default_flag   count    pct
0             0  982216  78.79
1             1  264485  21.21


In [11]:
con.close()
print("Validation complete ✓")

Validation complete ✓
